# CYB-5 — LLM zéro-shot local (Ollama) · pilote exploratoire

Lanceur Colab du script `src/evaluation/llm_pilot_eval.py`. Aucun entraînement. Les 160 emails ne sont **jamais** envoyés à un service externe : Ollama tourne dans cette session.

**Statut : pilote exploratoire à étiquettes de corpus** (80 Nazario positifs / 80 CEAS négatifs, confondus avec la source ; attaques non certifiées).

Avant de lancer : *Exécution → Modifier le type d'exécution → GPU T4*. Le dépôt `CYB5` (sans `data/`) doit être sur Drive dans `MyDrive/CYB5`. Le manifeste privé est téléversé à la main et n'est **pas** copié sur Drive. Ce notebook est livré sans sorties ; ne pas committer une copie exécutée.

In [ ]:
from pathlib import Path
import os, subprocess, json
from google.colab import drive
drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/CYB5")
for rel in ["src/models/llm_detector.py", "src/evaluation/llm_pilot_eval.py",
            "src/evaluation/experiment38_distilbert_pilot.py",
            "configs/llm_config_v1.json", "configs/llm_prompt_v1.txt"]:
    assert (ROOT / rel).is_file(), f"Fichier manquant dans le dépôt Drive : {rel}"
os.chdir(ROOT)
CFG = json.loads(Path("configs/llm_config_v1.json").read_text())
print("Modèle figé :", CFG["model"])

## 1. Ollama et modèle (seuls les poids sont téléchargés)

In [ ]:
%%bash
apt-get -qq install -y zstd pciutils > /dev/null 2>&1 || true
command -v ollama > /dev/null || curl -fsSL https://ollama.com/install.sh | sh
ollama --version

In [ ]:
import subprocess, time, urllib.request
def up():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434", timeout=2); return True
    except Exception:
        return False
if not up():
    subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(90):
        if up(): break
        time.sleep(1)
assert up(), "Ollama ne répond pas (voir /tmp/ollama.log)"
subprocess.run(["ollama", "pull", CFG["model"]], check=True)

## 2. Validation synthétique (14 messages inertes, non liés au pilote)
Vérifie le parsing JSON, le taux d'invalides et deux cas d'injection. Si le prompt doit changer, créer `llm_prompt_v2.txt` + une config v2 : v1 reste figé.

In [ ]:
!python -m src.evaluation.llm_pilot_eval --mode validate

## 3. Manifeste privé original (téléversement manuel)
Sélectionner **le fichier existant** `pilot_manifest.csv` (Exp. 38/41). Le script refuse un échantillon régénéré (couverture attendue 42 / 80 / 13).

In [ ]:
from google.colab import files
MANIFEST = Path("/content/pilot_manifest.csv")
if not MANIFEST.is_file():
    up_ = files.upload()
    assert "pilot_manifest.csv" in up_, "Choisir le fichier nommé pilot_manifest.csv"
    MANIFEST.write_bytes(up_["pilot_manifest.csv"])
print("Manifeste prêt (taille %d octets) — contenu non affiché." % MANIFEST.stat().st_size)

## 4. Évaluation : 160 messages × 4 conditions
Reprise automatique en cas de déconnexion (relancer la cellule). Les prédictions avec IDs restent dans `data/audit/cyb5_llm_pilot/` (privé, ignoré par git).

In [ ]:
!python -m src.evaluation.llm_pilot_eval --mode run --manifest /content/pilot_manifest.csv

## 5. Agrégat anonymisé (relire avant de le committer)

In [ ]:
import pandas as pd
agg = pd.read_csv("results/metrics/llm_aggregate.csv")
assert "sample_id" not in agg.columns
pd.set_option("display.width", 200)
print(agg[["condition","eligible_positive","tp","fp","tn","fn","f1","recall","fpr","delta_f1_vs_clean",
           "asr_denominator","detected_to_missed_flips","invalid_rate","latency_mean_s"]].round(4).to_string(index=False))

Fichiers à relire puis committer : `results/metrics/llm_aggregate.csv`, `results/metrics/llm_run_record.json`. **Ne jamais committer** `data/` ni le manifeste.